# Foundation Lab 6: AI Co-Scientist — Multi-Agent Scientific Research

**Series**: Agentic AI Science Playbook — Foundation  
**Goal**: Build a team of specialized AI agents that collaborate to conduct scientific research — from literature review to hypothesis generation to experiment design.  
**Prerequisites**: Foundation Labs 0–5 (especially Lab 4: LangGraph).  
**Time**: ~90 min.

## Learning Objectives

By the end of this lab, you will be able to:

- **Design a multi-agent system** where specialized agents collaborate on a shared scientific task
- **Implement the Orchestrator pattern** — a coordinator that delegates work and aggregates results
- **Use structured output schemas** (Pydantic) to enforce contracts between agents
- **Build four domain-specific agents**: Literature, Hypothesis, Critic, and Experiment
- **Evaluate multi-agent output quality** by comparing it to single-agent baselines

## Why This Matters

In January 2025, Google DeepMind unveiled **AI Co-Scientist** — a multi-agent system where specialized AI agents collaborate to generate novel research hypotheses, with results validated in wet-lab experiments for drug repurposing and novel target discovery. The key insight: **no single agent is an expert at everything**, but a *team* of focused agents can outperform any individual.

This pattern is appearing across the industry:

- **Google AI Co-Scientist**: Literature review + hypothesis generation + evaluation agents collaborating on biomedical research
- **NVIDIA multi-agent drug discovery**: Molecular generation, docking, ADMET scoring, and synthesis planning agents in a coordinated pipeline
- **Microsoft AutoGen**: A framework for multi-agent conversations where agents debate and refine solutions
- **CrewAI**: Role-based agent teams with delegation and collaboration protocols

The common thread: **decompose a complex task into specialized roles, then orchestrate their collaboration**. This is the pattern you will build in this lab.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Labs 0–5 (especially Lab 4: LangGraph graphs & cycles) |
| Libraries | `openai`, `pydantic` |
| Concepts | Structured output, tool calling, state machines |

**NVIDIA Connection**: The multi-agent orchestration pattern you build here maps directly to production systems built with:

- **NeMo Agent Toolkit** — NVIDIA's framework for building and deploying agent teams with NIM endpoints
- **NemoClaw** — Multi-agent coordination for robotics and physical AI
- **CrewAI + NIM** — Role-based agent teams powered by NVIDIA inference microservices

The agent roles in this lab (Literature, Hypothesis, Critic, Experiment) mirror real roles in NVIDIA BioNeMo-powered drug discovery pipelines, where specialized models handle molecular generation, property prediction, docking, and synthesis planning.

## Background: What Is an AI Co-Scientist?

An AI Co-Scientist is a **team of specialized agents** that collaborates on scientific research, mimicking the division of labor in a real research group. Each agent has a focused role:

| Agent | Role | Real-World Analogy |
|-------|------|--------------------|
| **Literature Agent** | Searches and summarizes relevant papers | Research assistant doing a lit review |
| **Hypothesis Agent** | Generates novel hypotheses based on literature gaps | Principal investigator brainstorming |
| **Critic Agent** | Evaluates hypotheses for novelty, feasibility, impact | Peer reviewer |
| **Experiment Agent** | Designs experiments to test top hypotheses | Lab manager writing protocols |

### Multi-Agent Flow

```
User Research Question
        |
        v
+-------------------+
|   ORCHESTRATOR     | <-- Coordinates the team
+--------+----------+
         |
         v
+--------+----------+
| Literature Agent   | --> Searches papers, finds gaps
+--------+----------+
         |
         v
+--------+----------+
| Hypothesis Agent   | --> Proposes novel hypotheses
+--------+----------+
         |
         v
+--------+----------+
|  Critic Agent      | --> Scores novelty, feasibility, impact
+--------+----------+
         |
         v
+--------+----------+
| Experiment Agent   | --> Designs protocols to test top hypotheses
+-------------------+
```

Each agent receives context from the previous agents and produces **structured output** that feeds forward. The Orchestrator coordinates the flow and aggregates the final result.

## Concept: Multi-Agent vs Single-Agent

Why use four agents instead of one big prompt? The difference matters:

| Dimension | Single Agent (one prompt) | Multi-Agent (specialized team) |
|-----------|--------------------------|-------------------------------|
| **Prompt complexity** | One massive prompt tries to do everything | Each agent has a focused, manageable prompt |
| **Output quality** | Tends to be shallow across all tasks | Deep expertise in each sub-task |
| **Structured output** | Hard to enforce a single schema for everything | Each agent has its own Pydantic contract |
| **Debuggability** | One black box — hard to see what went wrong | Inspect each agent's output independently |
| **Modularity** | Change one thing, rewrite everything | Swap one agent without touching others |
| **Scalability** | Limited by context window | Each agent uses only the context it needs |
| **Error isolation** | One bad output corrupts everything | A failing agent can be retried independently |

The trade-off: multi-agent systems require **more LLM calls** (latency, cost) and **inter-agent contracts** (schemas). The payoff is dramatically better quality and maintainability for complex tasks.

## Concept: The Orchestrator Pattern

The **Orchestrator** is the coordinator that ties the agents together. It does not do scientific work itself — instead, it:

1. **Receives** the user's research question
2. **Delegates** to each specialist agent in the right order
3. **Passes context** from one agent to the next (literature feeds into hypothesis generation, etc.)
4. **Aggregates** the final result into a coherent research report

This is different from a "chat-based" multi-agent system (like AutoGen) where agents debate freely. Our orchestrator follows a **pipeline pattern** — a directed flow with clear handoffs. This is simpler to debug and more predictable for scientific workflows.

```
Orchestrator = Pipeline Coordinator
    |
    +--> Step 1: Call literature_agent(question)       --> LiteratureReport
    +--> Step 2: Call hypothesis_agent(question, lit)   --> HypothesisReport
    +--> Step 3: Call critic_agent(hypotheses, lit)     --> CriticReport
    +--> Step 4: Call experiment_agent(top_hyp, crit)   --> ExperimentDesign
    |
    +--> Return aggregated {literature, hypotheses, critique, experiment}
```

In production, you could replace this linear pipeline with a LangGraph state machine (Lab 4) that includes cycles — for example, sending rejected hypotheses back to the Hypothesis Agent for revision.

---

## Part 2: Setup

### Install Dependencies
We need `openai` for LLM access and `pydantic` for structured output schemas that enforce contracts between agents.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Standard dual-provider setup. Detects whether to use NVIDIA NIM or OpenAI based on environment variables.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os, json
from getpass import getpass
from openai import OpenAI

use_nim = os.environ.get("USE_NIM", "").lower() in ("1", "true", "yes") or "NIM_API_KEY" in os.environ

if use_nim:
    if "NIM_API_KEY" not in os.environ:
        os.environ["NIM_API_KEY"] = getpass("NVIDIA NIM API key: ")
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ["NIM_API_KEY"],
    )
    MODEL = os.environ.get("NIM_MODEL", "nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
    client = OpenAI()
    MODEL = "gpt-4o-mini"

print(f"Using model: {MODEL}")

---

## Part 3: Define the Agent Team

Each agent is a **Python function** that calls the LLM with a specialized prompt and returns **structured output** via a Pydantic schema. The schemas act as contracts — they guarantee that one agent's output matches the next agent's expected input.

We will define:
1. **Output schemas** — Pydantic models for each agent's response
2. **Agent functions** — each calls the LLM, parses the response, and returns a typed object
3. **An LLM helper** — shared utility for making API calls with JSON output

### Schema: Literature Agent Output
The Literature Agent returns a structured report with papers found, key findings, research gaps, and an overall summary. Each field is documented with a `Field` description.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional


class LiteratureReport(BaseModel):
    """Output schema for the Literature Agent."""
    papers_found: list[str] = Field(description="Key papers found (title + authors + year)")
    key_findings: list[str] = Field(description="Main findings from the literature")
    research_gaps: list[str] = Field(description="Identified gaps in current research")
    summary: str = Field(description="Overall literature summary (2-3 sentences)")


print("LiteratureReport fields:")
for name, field in LiteratureReport.model_fields.items():
    print(f"  {name}: {field.annotation.__name__ if hasattr(field.annotation, '__name__') else field.annotation}")

### Schema: Hypothesis Agent Output
The Hypothesis Agent produces multiple hypotheses, each with a title, description, rationale, self-assessed novelty score, and testability note. These are bundled in a `HypothesisReport`.

In [ ]:
class Hypothesis(BaseModel):
    """A single research hypothesis."""
    title: str = Field(description="Short descriptive title")
    description: str = Field(description="What the hypothesis proposes")
    rationale: str = Field(description="Why this hypothesis is plausible")
    novelty_score: int = Field(ge=1, le=10, description="Self-assessed novelty 1-10")
    testability: str = Field(description="How this could be experimentally tested")


class HypothesisReport(BaseModel):
    """Output schema for the Hypothesis Agent."""
    hypotheses: list[Hypothesis] = Field(description="Generated hypotheses")
    reasoning: str = Field(description="Overall reasoning behind the hypothesis set")


print(f"Hypothesis has {len(Hypothesis.model_fields)} fields")
print(f"HypothesisReport wraps a list of Hypothesis objects + reasoning")

### Schema: Critic Agent Output
The Critic Agent evaluates each hypothesis on three dimensions (novelty, feasibility, impact) and gives a recommendation: pursue, revise, or reject. It also identifies the top hypothesis.

In [ ]:
class HypothesisCritique(BaseModel):
    """Critique of a single hypothesis."""
    hypothesis_title: str
    novelty: int = Field(ge=1, le=10, description="Novelty score 1-10")
    feasibility: int = Field(ge=1, le=10, description="Feasibility score 1-10")
    impact: int = Field(ge=1, le=10, description="Potential impact score 1-10")
    strengths: list[str] = Field(description="Key strengths")
    weaknesses: list[str] = Field(description="Key weaknesses")
    overall_score: float = Field(description="Weighted average score")
    recommendation: str = Field(description="One of: pursue, revise, reject")


class CriticReport(BaseModel):
    """Output schema for the Critic Agent."""
    critiques: list[HypothesisCritique] = Field(description="Critique for each hypothesis")
    top_hypothesis: str = Field(description="Title of the highest-scored hypothesis")


print(f"HypothesisCritique scores: novelty, feasibility, impact -> overall_score")
print(f"Recommendations: pursue | revise | reject")

### Schema: Experiment Agent Output
The Experiment Agent produces a full experimental protocol: methodology, expected outcomes, resources, timeline, and success criteria.

In [ ]:
class ExperimentDesign(BaseModel):
    """Output schema for the Experiment Agent."""
    hypothesis_tested: str = Field(description="Which hypothesis this experiment tests")
    experiment_type: str = Field(description="Type of experiment (e.g., RCT, in vitro, computational)")
    methodology: str = Field(description="Detailed methodology description")
    expected_outcomes: list[str] = Field(description="What outcomes to expect")
    required_resources: list[str] = Field(description="Equipment, reagents, data needed")
    timeline: str = Field(description="Estimated timeline")
    success_criteria: str = Field(description="How to determine if the experiment succeeded")


print("ExperimentDesign fields:")
for name in ExperimentDesign.model_fields:
    print(f"  - {name}")

### LLM Helper: JSON-Mode Calls
A shared utility that sends a prompt to the LLM with `response_format=json_object` and returns the parsed dictionary. All agents use this to get structured responses.

In [ ]:
def llm_json_call(prompt: str, temperature: float = 0.3, max_tokens: int = 1500) -> dict:
    """Call the LLM with JSON output mode and return parsed dict."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content or "{}"
    return json.loads(raw)


# Quick test
test = llm_json_call("Return JSON: {\"status\": \"ok\"}", max_tokens=50)
print("LLM JSON helper works:", test)

### Agent 1: Literature Agent
The Literature Agent receives a research question and returns a structured literature report. It simulates searching the scientific literature by using the LLM's training knowledge as a proxy for a real search API.

> **Production note**: In a real system, this agent would call PubMed, Semantic Scholar, or NVIDIA BioNeMo's literature search APIs. Here we use the LLM's knowledge as a stand-in.

In [ ]:
def literature_agent(research_question: str) -> LiteratureReport:
    """Search the literature and return a structured report."""
    schema_hint = json.dumps(LiteratureReport.model_json_schema(), indent=2)

    prompt = f"""You are a scientific literature review specialist.
Your task is to search the literature on the following research question and provide a structured report.

Research question: {research_question}

Provide:
1. papers_found: 3-5 key relevant papers (include title, authors, year)
2. key_findings: 3-5 main findings from the literature
3. research_gaps: 2-3 open gaps in current research
4. summary: A 2-3 sentence overall summary

Return your response as JSON matching this schema:
{schema_hint}"""

    data = llm_json_call(prompt, temperature=0.3, max_tokens=1500)
    return LiteratureReport(**data)


print("literature_agent() defined -- takes a research question, returns LiteratureReport")

### Agent 2: Hypothesis Agent
The Hypothesis Agent receives the research question **and** the literature report. It reads the gaps identified by the Literature Agent and generates novel hypotheses that address those gaps. This is where the "creative" science happens.

In [ ]:
def hypothesis_agent(research_question: str, literature: LiteratureReport) -> HypothesisReport:
    """Generate novel hypotheses based on the literature review."""
    schema_hint = json.dumps(HypothesisReport.model_json_schema(), indent=2)

    prompt = f"""You are a creative scientific hypothesis generator.
Based on the research question and literature review below, generate 2-3 novel hypotheses.

Research question: {research_question}

Literature summary: {literature.summary}
Key findings: {json.dumps(literature.key_findings)}
Research gaps: {json.dumps(literature.research_gaps)}

For each hypothesis, provide:
- title: A short descriptive title
- description: What the hypothesis proposes
- rationale: Why this is plausible given the literature
- novelty_score: 1-10 self-assessment of novelty
- testability: How it could be experimentally tested

Also provide overall reasoning for why you chose these hypotheses.

Return as JSON matching this schema:
{schema_hint}"""

    data = llm_json_call(prompt, temperature=0.7, max_tokens=2000)
    return HypothesisReport(**data)


print("hypothesis_agent() defined -- takes question + literature, returns HypothesisReport")

### Agent 3: Critic Agent
The Critic Agent is the "peer reviewer." It receives the hypotheses **and** the literature, then scores each hypothesis on novelty, feasibility, and impact. It also identifies strengths, weaknesses, and gives a recommendation (pursue / revise / reject).

In [ ]:
def critic_agent(hypotheses: HypothesisReport, literature: LiteratureReport) -> CriticReport:
    """Evaluate and rank hypotheses like a peer reviewer."""
    schema_hint = json.dumps(CriticReport.model_json_schema(), indent=2)

    hyp_descriptions = []
    for h in hypotheses.hypotheses:
        hyp_descriptions.append(
            f"- {h.title}: {h.description} (rationale: {h.rationale})"
        )

    prompt = f"""You are a rigorous scientific peer reviewer and critic.
Evaluate each hypothesis below against the literature. Be constructive but honest.

Literature summary: {literature.summary}
Research gaps: {json.dumps(literature.research_gaps)}

Hypotheses to evaluate:
{chr(10).join(hyp_descriptions)}

For each hypothesis, score:
- novelty (1-10): How original is this idea?
- feasibility (1-10): Can this realistically be tested with current technology?
- impact (1-10): If true, how significant would this be?
- strengths: 2-3 key strengths
- weaknesses: 1-2 key weaknesses
- overall_score: weighted average (novelty*0.3 + feasibility*0.3 + impact*0.4)
- recommendation: "pursue", "revise", or "reject"

Also identify the top_hypothesis (highest overall_score).

Return as JSON matching this schema:
{schema_hint}"""

    data = llm_json_call(prompt, temperature=0.2, max_tokens=2000)
    return CriticReport(**data)


print("critic_agent() defined -- takes hypotheses + literature, returns CriticReport")

### Agent 4: Experiment Agent
The Experiment Agent receives the top-ranked hypothesis and the critic's feedback, then designs a detailed experimental protocol to test it. This is the "lab manager" of the team.

In [ ]:
def experiment_agent(top_hypothesis: str, critique: CriticReport) -> ExperimentDesign:
    """Design an experiment to test the top-ranked hypothesis."""
    schema_hint = json.dumps(ExperimentDesign.model_json_schema(), indent=2)

    # Find the critique for the top hypothesis
    top_critique = None
    for c in critique.critiques:
        if c.hypothesis_title == top_hypothesis:
            top_critique = c
            break

    critique_context = ""
    if top_critique:
        critique_context = f"""
Critic feedback:
- Strengths: {json.dumps(top_critique.strengths)}
- Weaknesses: {json.dumps(top_critique.weaknesses)}
- Feasibility score: {top_critique.feasibility}/10
"""

    prompt = f"""You are an experimental design specialist.
Design a detailed experiment to test the following hypothesis.

Hypothesis: {top_hypothesis}
{critique_context}

Provide:
- hypothesis_tested: The hypothesis being tested
- experiment_type: Type of experiment (e.g., randomized controlled trial, in vitro assay, computational simulation)
- methodology: Detailed step-by-step methodology (3-5 sentences)
- expected_outcomes: 2-3 expected outcomes if the hypothesis is correct
- required_resources: 3-5 resources needed (equipment, reagents, datasets, etc.)
- timeline: Estimated timeline (e.g., "6-12 months")
- success_criteria: How to determine if the experiment supports or refutes the hypothesis

Return as JSON matching this schema:
{schema_hint}"""

    data = llm_json_call(prompt, temperature=0.3, max_tokens=1500)
    return ExperimentDesign(**data)


print("experiment_agent() defined -- takes top hypothesis + critique, returns ExperimentDesign")

---

## Part 4: The Orchestrator

Now we wire the four agents into a single **orchestrator function** that runs the full AI Co-Scientist pipeline. The orchestrator:
1. Takes a research question from the user
2. Calls each agent in sequence, passing context forward
3. Prints progress at each step
4. Returns the aggregated results from all agents

In [ ]:
def ai_coscientist(research_question: str) -> dict:
    """Run the full AI Co-Scientist pipeline."""
    print("=" * 60)
    print(f"  AI CO-SCIENTIST")
    print(f"  Question: {research_question}")
    print("=" * 60)

    # Step 1: Literature Review
    print("\n[Step 1/4] Literature Agent searching...")
    lit_report = literature_agent(research_question)
    print(f"  -> Found {len(lit_report.papers_found)} papers")
    print(f"  -> Identified {len(lit_report.research_gaps)} research gaps")

    # Step 2: Hypothesis Generation
    print("\n[Step 2/4] Hypothesis Agent generating ideas...")
    hyp_report = hypothesis_agent(research_question, lit_report)
    print(f"  -> Generated {len(hyp_report.hypotheses)} hypotheses")
    for h in hyp_report.hypotheses:
        print(f"     - {h.title} (self-rated novelty: {h.novelty_score}/10)")

    # Step 3: Critique & Ranking
    print("\n[Step 3/4] Critic Agent evaluating...")
    critic_report = critic_agent(hyp_report, lit_report)
    print(f"  -> Top hypothesis: {critic_report.top_hypothesis}")
    for c in critic_report.critiques:
        print(f"     - {c.hypothesis_title}: {c.overall_score:.1f}/10 ({c.recommendation})")

    # Step 4: Experiment Design
    print("\n[Step 4/4] Experiment Agent designing protocol...")
    experiment = experiment_agent(critic_report.top_hypothesis, critic_report)
    print(f"  -> Experiment type: {experiment.experiment_type}")
    print(f"  -> Timeline: {experiment.timeline}")

    print("\n" + "=" * 60)
    print("  PIPELINE COMPLETE")
    print("=" * 60)

    return {
        "question": research_question,
        "literature": lit_report,
        "hypotheses": hyp_report,
        "critique": critic_report,
        "experiment": experiment,
    }


print("ai_coscientist() orchestrator defined -- ready to run!")

### Helper: Pretty-Print Results
A utility function to display each agent's output in a readable format after the pipeline runs.

In [ ]:
def print_full_report(result: dict):
    """Pretty-print the full AI Co-Scientist report."""
    lit = result["literature"]
    hyp = result["hypotheses"]
    crit = result["critique"]
    exp = result["experiment"]

    # Literature
    print("\n" + "=" * 60)
    print("  LITERATURE REVIEW")
    print("=" * 60)
    print(f"\nSummary: {lit.summary}\n")
    print("Papers found:")
    for i, p in enumerate(lit.papers_found, 1):
        print(f"  {i}. {p}")
    print("\nKey findings:")
    for f in lit.key_findings:
        print(f"  * {f}")
    print("\nResearch gaps:")
    for g in lit.research_gaps:
        print(f"  ! {g}")

    # Hypotheses
    print("\n" + "=" * 60)
    print("  HYPOTHESES")
    print("=" * 60)
    print(f"\nReasoning: {hyp.reasoning}\n")
    for i, h in enumerate(hyp.hypotheses, 1):
        print(f"Hypothesis {i}: {h.title}")
        print(f"  Description: {h.description}")
        print(f"  Rationale:   {h.rationale}")
        print(f"  Novelty:     {h.novelty_score}/10")
        print(f"  Testability: {h.testability}")
        print()

    # Critiques
    print("=" * 60)
    print("  CRITIC EVALUATION")
    print("=" * 60)
    for c in crit.critiques:
        print(f"\n{c.hypothesis_title}")
        print(f"  Novelty: {c.novelty}/10 | Feasibility: {c.feasibility}/10 | Impact: {c.impact}/10")
        print(f"  Overall: {c.overall_score:.1f}/10 -> {c.recommendation.upper()}")
        print(f"  Strengths:  {', '.join(c.strengths)}")
        print(f"  Weaknesses: {', '.join(c.weaknesses)}")
    print(f"\n  >>> Top hypothesis: {crit.top_hypothesis}")

    # Experiment
    print("\n" + "=" * 60)
    print("  EXPERIMENT DESIGN")
    print("=" * 60)
    print(f"\nHypothesis tested: {exp.hypothesis_tested}")
    print(f"Experiment type:   {exp.experiment_type}")
    print(f"Timeline:          {exp.timeline}")
    print(f"\nMethodology:\n  {exp.methodology}")
    print(f"\nExpected outcomes:")
    for o in exp.expected_outcomes:
        print(f"  - {o}")
    print(f"\nRequired resources:")
    for r in exp.required_resources:
        print(f"  - {r}")
    print(f"\nSuccess criteria:\n  {exp.success_criteria}")


print("print_full_report() defined")

---

## Part 5: Experiments

Now we put the AI Co-Scientist to work on real research questions across different scientific domains. Each experiment demonstrates the full 4-agent pipeline on a different topic.

### Experiment 1: Antibiotic Resistance
A pressing public health challenge. Let's see how the Co-Scientist team analyzes mechanisms of antibiotic resistance in hospital-acquired infections.

In [ ]:
result_1 = ai_coscientist(
    "What are the mechanisms of antibiotic resistance in hospital-acquired infections?"
)

<details>
<summary>Expected output (click to expand)</summary>

```
============================================================
AI CO-SCIENTIST PIPELINE
============================================================
Research Question:
  What are the mechanisms of antibiotic resistance in hospital-acquired infections?

[1/4] Literature Agent — searching...
[2/4] Hypothesis Agent — generating hypotheses...
[3/4] Critic Agent — evaluating hypotheses...
[4/4] Experiment Agent — designing experiment...

PIPELINE COMPLETE
============================================================
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The pipeline always runs all 4 agents in sequence.
</details>

In [ ]:
print_full_report(result_1)

<details>
<summary>Expected output (click to expand)</summary>

```
========== LITERATURE REPORT ==========
Papers found: 5
Key findings:
  1. Horizontal gene transfer via plasmids is the dominant mechanism...
  2. Biofilm formation on medical devices creates protected environments...
  3. Efflux pumps provide broad-spectrum resistance to multiple drug classes...
Research gaps:
  - Limited understanding of inter-species resistance transfer in ICU settings
  - Few studies on real-time resistance monitoring

========== HYPOTHESES ==========
Hypothesis 1: Biofilm-mediated horizontal gene transfer accelerates...
  Novelty: 7/10 | Feasibility: 6/10 | Impact: 8/10

Hypothesis 2: Sub-inhibitory antibiotic concentrations in hospital...
  Novelty: 6/10 | Feasibility: 7/10 | Impact: 7/10

========== CRITIC REPORT ==========
Top hypothesis: [selected by critic based on scores]
  Novelty: 7/10 | Feasibility: 6/10 | Impact: 8/10
  Key concern: Requires longitudinal sampling protocol...

========== EXPERIMENT DESIGN ==========
Hypothesis tested: [top-ranked hypothesis]
Methodology: ...
Expected outcomes: ...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What the Co-Scientist Did (Experiment 1)

Notice how the agents built on each other's work:

1. **Literature Agent** identified key papers on resistance mechanisms (efflux pumps, beta-lactamases, biofilm formation) and spotted gaps in understanding cross-species horizontal gene transfer in hospital environments.
2. **Hypothesis Agent** read those gaps and proposed novel hypotheses that specifically address them — not generic ideas, but gap-targeted ones.
3. **Critic Agent** independently evaluated each hypothesis, often scoring the self-assessed novelty differently than the Hypothesis Agent did. This "second opinion" catches overconfidence.
4. **Experiment Agent** designed a protocol tailored to the top-ranked hypothesis, incorporating the Critic's feasibility feedback.

This is the power of multi-agent: **each agent sees the problem through a different lens**.

### Experiment 2: CRISPR for Sickle Cell Disease
A different domain — gene therapy. This tests whether the same agent team can handle a genetics/therapeutics question just as well as an infectious disease question.

In [ ]:
result_2 = ai_coscientist(
    "How can CRISPR be used to treat sickle cell disease?"
)

<details>
<summary>Expected output (click to expand)</summary>

```
============================================================
AI CO-SCIENTIST PIPELINE
============================================================
Research Question:
  How can CRISPR be used to treat sickle cell disease?

[1/4] Literature Agent — searching...
[2/4] Hypothesis Agent — generating hypotheses...
[3/4] Critic Agent — evaluating hypotheses...
[4/4] Experiment Agent — designing experiment...

PIPELINE COMPLETE
============================================================
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The pipeline always runs all 4 agents in sequence.
</details>

In [ ]:
print_full_report(result_2)

<details>
<summary>Expected output (click to expand)</summary>

```
========== LITERATURE REPORT ==========
Papers found: 5
Key findings:
  1. CRISPR-Cas9 editing of BCL11A enhancer in HSCs reactivates fetal hemoglobin...
  2. Vertex/CRISPR's exa-cel received FDA approval in Dec 2023...
  3. Base editing approaches show promise for direct HBB gene correction...

========== HYPOTHESES ==========
Hypothesis 1: Prime editing of the HBB gene for precise sickle mutation correction...
Hypothesis 2: Dual-guide CRISPR approach targeting both BCL11A and HBB simultaneously...

========== CRITIC REPORT ==========
Top hypothesis: [selected based on novelty, feasibility, and impact scores]

========== EXPERIMENT DESIGN ==========
Hypothesis tested: [top-ranked hypothesis]
Methodology: ...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What the Co-Scientist Did (Experiment 2)

The same four agents, but a completely different scientific domain:

1. **Literature Agent** found papers on CRISPR-Cas9 editing of the BCL11A enhancer, Casgevy (the first approved CRISPR therapy), and fetal hemoglobin reactivation strategies.
2. **Hypothesis Agent** generated hypotheses that go *beyond* the current literature — perhaps proposing novel delivery mechanisms, combination approaches, or in vivo editing strategies.
3. **Critic Agent** evaluated feasibility differently here because gene therapy experiments are expensive and heavily regulated — practical constraints matter.
4. **Experiment Agent** adapted the protocol to the gene therapy domain (cell culture, animal models, delivery vectors).

Key insight: The agents are **domain-agnostic** — the same architecture works for infectious disease, gene therapy, or any other scientific field. The specialization comes from the prompts, not the code.

### Experiment 3: Gut Microbiome and Alzheimer's Disease
A cross-disciplinary question that bridges microbiology and neuroscience. This is where the Co-Scientist pattern really shines — connecting dots across fields that a single specialist might miss.

In [ ]:
result_3 = ai_coscientist(
    "What role does gut microbiome play in Alzheimer's disease?"
)

<details>
<summary>Expected output (click to expand)</summary>

```
============================================================
AI CO-SCIENTIST PIPELINE
============================================================
Research Question:
  What role does gut microbiome play in Alzheimer's disease?

[1/4] Literature Agent — searching...
[2/4] Hypothesis Agent — generating hypotheses...
[3/4] Critic Agent — evaluating hypotheses...
[4/4] Experiment Agent — designing experiment...

PIPELINE COMPLETE
============================================================
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The pipeline always runs all 4 agents in sequence.
</details>

In [ ]:
print_full_report(result_3)

<details>
<summary>Expected output (click to expand)</summary>

```
========== LITERATURE REPORT ==========
Papers found: 5
Key findings:
  1. Gut-brain axis communication via vagus nerve and microbial metabolites...
  2. Altered microbiome composition (dysbiosis) observed in AD patients...
  3. Short-chain fatty acids (SCFAs) may modulate neuroinflammation...

========== HYPOTHESES ==========
Hypothesis 1: Specific SCFA-producing bacteria protect against neuroinflammation...
Hypothesis 2: Microbiome-derived amyloid proteins cross-seed A-beta aggregation...

========== CRITIC REPORT ==========
Top hypothesis: [selected based on scores]

========== EXPERIMENT DESIGN ==========
Hypothesis tested: [top-ranked hypothesis]
Methodology: Germ-free mouse model with defined microbial communities...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What the Co-Scientist Did (Experiment 3)

This cross-disciplinary question illustrates a key advantage of the multi-agent approach:

1. **Literature Agent** had to bridge two fields — finding papers from both microbiome research and Alzheimer's neuroscience. The gut-brain axis is a rapidly evolving field with lots of recent work.
2. **Hypothesis Agent** could propose connections that span disciplines: metabolite signaling, immune modulation, amyloid cross-seeding from microbial proteins.
3. **Critic Agent** had to assess feasibility across both fields — microbiome experiments have different constraints than neuroscience ones.
4. **Experiment Agent** might propose a multi-disciplinary protocol involving both germ-free mouse models and neuroimaging.

This is exactly the kind of question where AI Co-Scientists add the most value: **finding connections across siloed knowledge domains**.

---

## Part 6: Detailed Output Exploration

Let's dig deeper into the structured outputs to see the richness of information the multi-agent pipeline produces. We will use Experiment 1 (antibiotic resistance) as our example.

### Literature Report: Papers and Gaps

In [ ]:
# Deep dive into the Literature Agent's output
lit = result_1["literature"]

print("LITERATURE AGENT - Structured Output")
print("=" * 50)
print(f"\nType: {type(lit).__name__}")
print(f"Fields: {list(lit.model_fields.keys())}")
print(f"\nPapers found ({len(lit.papers_found)}):")
for i, paper in enumerate(lit.papers_found, 1):
    print(f"  [{i}] {paper}")
print(f"\nResearch gaps ({len(lit.research_gaps)}):")
for gap in lit.research_gaps:
    print(f"  >> {gap}")

<details>
<summary>Expected output (click to expand)</summary>

```
LITERATURE AGENT - Structured Output
=====================================
Papers found  : 5
Key findings  : 3
Research gaps  : 2

Papers:
  1. [2024] Horizontal gene transfer dynamics in ICU environments...
  2. [2023] Biofilm-mediated antibiotic resistance: mechanisms and...
  ...

Key Findings:
  1. Horizontal gene transfer via plasmids is the dominant mechanism...
  2. Biofilm formation on medical devices creates protected environments...
  3. Efflux pumps provide broad-spectrum resistance...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Hypothesis Ranking: Critic Scores Compared
Let's build a comparison table of all hypotheses with their critic scores side-by-side.

In [ ]:
# Compare hypothesis self-scores vs critic scores
crit = result_1["critique"]
hyp = result_1["hypotheses"]

print("HYPOTHESIS RANKING — Self-Score vs Critic Score")
print("=" * 70)
print(f"{'Title':<35} {'Self':>6} {'Nov':>5} {'Feas':>5} {'Imp':>5} {'Overall':>8} {'Rec':>8}")
print("-" * 70)

for h in hyp.hypotheses:
    # Find matching critique
    matching = [c for c in crit.critiques if c.hypothesis_title == h.title]
    if matching:
        c = matching[0]
        print(f"{h.title[:35]:<35} {h.novelty_score:>5}/10 {c.novelty:>4}/10 {c.feasibility:>4}/10 {c.impact:>4}/10 {c.overall_score:>7.1f} {c.recommendation:>8}")
    else:
        print(f"{h.title[:35]:<35} {h.novelty_score:>5}/10   (no matching critique found)")

print(f"\nTop hypothesis selected: {crit.top_hypothesis}")

<details>
<summary>Expected output (click to expand)</summary>

```
HYPOTHESIS COMPARISON: Self-Score vs Critic Score
===================================================
Hypothesis                     Self    Critic   Delta
------------------------------------------------------
Biofilm-mediated HGT...         7.0     6.7     -0.3
Sub-inhibitory concentr...      6.7     7.0     +0.3
Phage-bacteria co-evol...       6.3     5.7     -0.7

Top-ranked by critic: [hypothesis with highest critic score]
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Experiment Protocol: Full Detail
The Experiment Agent's output is a complete protocol that could be handed to a lab team. Let's examine it as structured data.

In [ ]:
# Experiment design as structured data
exp = result_1["experiment"]

print("EXPERIMENT DESIGN — Structured Protocol")
print("=" * 50)
print(f"\nHypothesis:      {exp.hypothesis_tested}")
print(f"Type:            {exp.experiment_type}")
print(f"Timeline:        {exp.timeline}")
print(f"Success criteria: {exp.success_criteria}")

print(f"\nMethodology:")
print(f"  {exp.methodology}")

print(f"\nExpected outcomes ({len(exp.expected_outcomes)}):")
for i, outcome in enumerate(exp.expected_outcomes, 1):
    print(f"  {i}. {outcome}")

print(f"\nRequired resources ({len(exp.required_resources)}):")
for resource in exp.required_resources:
    print(f"  - {resource}")

# Show as raw JSON to demonstrate structured output
print("\n\nRaw JSON (for programmatic access):")
print(json.dumps(exp.model_dump(), indent=2)[:500] + "...")

<details>
<summary>Expected output (click to expand)</summary>

```
EXPERIMENT DESIGN — Structured Protocol
========================================
Hypothesis tested : [top-ranked hypothesis]
Methodology       : Longitudinal sampling from ICU patients...
Expected outcomes : Identification of resistance gene transfer rates...
Timeline          : 12 months
Key variables     : Antibiotic exposure, biofilm presence, bacterial species
Controls          : Non-ICU ward samples, antibiotic-free controls
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Agent Interaction Trace
Let's visualize the data flow between agents — how many tokens of context each agent received and produced.

In [ ]:
# Agent interaction trace — data sizes flowing between agents
def count_chars(obj):
    """Rough measure of information volume."""
    return len(json.dumps(obj.model_dump()))

lit = result_1["literature"]
hyp = result_1["hypotheses"]
crit = result_1["critique"]
exp = result_1["experiment"]

print("AGENT INTERACTION TRACE")
print("=" * 55)
print()
print(f"  User Question")
print(f"       | ({len(result_1['question'])} chars)")
print(f"       v")
print(f"  Literature Agent --> LiteratureReport")
print(f"       | ({count_chars(lit)} chars: {len(lit.papers_found)} papers, {len(lit.research_gaps)} gaps)")
print(f"       v")
print(f"  Hypothesis Agent --> HypothesisReport")
print(f"       | ({count_chars(hyp)} chars: {len(hyp.hypotheses)} hypotheses)")
print(f"       v")
print(f"  Critic Agent     --> CriticReport")
print(f"       | ({count_chars(crit)} chars: {len(crit.critiques)} critiques)")
print(f"       v")
print(f"  Experiment Agent --> ExperimentDesign")
print(f"       | ({count_chars(exp)} chars: 1 protocol)")
print(f"       v")
print(f"  DONE")
print()
total = count_chars(lit) + count_chars(hyp) + count_chars(crit) + count_chars(exp)
print(f"Total structured output: {total:,} characters across 4 agents")
print(f"Total LLM calls: 4 (one per agent)")

<details>
<summary>Expected output (click to expand)</summary>

```
AGENT INTERACTION TRACE
========================
Literature Agent output : ~2,500 chars
  → fed into Hypothesis Agent

Hypothesis Agent output : ~3,200 chars
  → fed into Critic Agent (along with Literature)

Critic Agent output     : ~2,800 chars
  → fed into Experiment Agent

Experiment Agent output : ~1,500 chars

Total pipeline context  : ~10,000 chars across 4 agents
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

---

## Part 7: Reflection & Summary

### Concept: How Multi-Agent Improves Over Single-Agent

Consider what happens if we ask a single LLM the same research question in one shot:

```
"What are the mechanisms of antibiotic resistance in hospital-acquired 
infections? Find relevant papers, generate novel hypotheses, evaluate them, 
and design an experiment to test the best one."
```

A single call would produce a **wall of text** — a monologue that tries to do everything at once. In contrast, our 4-agent pipeline produces:

| Metric | Single Agent | Multi-Agent (4 agents) |
|--------|-------------|----------------------|
| **Structure** | Unstructured prose | Typed Pydantic objects |
| **Depth** | Shallow on each sub-task | Deep on each sub-task |
| **Critique** | Self-evaluates (biased) | Independent critic (less biased) |
| **Debuggability** | One black box | Inspect each agent separately |
| **Reusability** | Monolithic prompt | Swap/upgrade individual agents |
| **Temperature control** | One temperature for all | Low for lit review, high for hypotheses |

The multi-agent approach is especially valuable when:
- The task has **distinct sub-tasks** that benefit from different prompting strategies
- You need **independent evaluation** (the Critic should not be biased by the Hypothesis Agent's self-scores)
- You want to **iterate on one agent** without changing the others
- You need **structured, programmatically accessible** output at each stage

### Reflection Questions

1. **Agent specialization**: We used the same LLM for all four agents — only the prompts differ. How would the system change if you used different models for different agents (e.g., a smaller model for literature search, a larger model for hypothesis generation)?

2. **Feedback loops**: Our pipeline is linear (lit -> hyp -> critic -> experiment). What would happen if the Critic could send rejected hypotheses back to the Hypothesis Agent for revision? How would you implement this with LangGraph (Lab 4)?

3. **Real data integration**: The Literature Agent uses the LLM's training knowledge as a proxy for real search. How would you connect it to PubMed, Semantic Scholar, or arXiv APIs? What changes to the schema and prompt would be needed?

4. **Scaling the team**: What additional agents would be valuable? Consider: a **Synthesis Agent** (combines findings across experiments), a **Safety Agent** (checks for ethical concerns), or a **Cost Estimator Agent** (budgets the experiment).

5. **Human-in-the-loop**: At which point in the pipeline would human oversight be most valuable? After hypothesis generation? After the critic's evaluation? Before experiment execution?

### NVIDIA SDK Connection: Production Multi-Agent Systems

The patterns in this lab map directly to production multi-agent architectures:

| Lab Pattern | NVIDIA Production Equivalent |
|------------|------------------------------|
| Specialized agent functions | **NeMo Agent Toolkit** agents with NIM endpoints |
| Pydantic output schemas | **Guardrails** for structured output validation |
| Orchestrator pipeline | **NemoClaw** multi-agent coordination |
| Literature Agent | **BioNeMo** literature search + embedding retrieval |
| Hypothesis Agent | **NIM** inference with domain-tuned models |
| Critic Agent | **LLM-as-Judge** pattern (Lab 5) at scale |
| Experiment Agent | **BioNeMo** experiment planning + simulation setup |

**Key NVIDIA tools for production multi-agent systems:**

- **NeMo Agent Toolkit**: Build, deploy, and manage teams of agents with NIM-powered inference. Each agent can use a different NIM endpoint optimized for its role.
- **NemoClaw**: Orchestrate multi-agent workflows for physical AI — robots that collaborate using the same delegation patterns we built here.
- **CrewAI + NIM**: Define agent teams with roles, goals, and delegation rules. CrewAI handles the orchestration; NIM provides the inference.
- **NVIDIA AI Blueprints**: Pre-built multi-agent workflows for drug discovery, climate modeling, and digital biology that you can customize.

## Summary

| Concept | What You Built | Key Takeaway |
|---------|---------------|--------------|
| **Multi-agent architecture** | 4 specialized agents | Decompose complex tasks into focused roles |
| **Pydantic contracts** | Typed schemas for each agent | Structured output enables reliable handoffs |
| **Literature Agent** | Searches and summarizes papers | Domain knowledge extraction |
| **Hypothesis Agent** | Generates gap-targeted hypotheses | Creative ideation from structured input |
| **Critic Agent** | Independent evaluation with scores | Reduces self-assessment bias |
| **Experiment Agent** | Designs testable protocols | Actionable output from abstract ideas |
| **Orchestrator pattern** | Pipeline coordinator function | Delegates, passes context, aggregates |
| **Temperature tuning** | Different temps per agent | Low for factual, high for creative tasks |

### What You Can Do Next

- **Lab 4 extension**: Convert the orchestrator to a LangGraph state machine with retry cycles (send rejected hypotheses back for revision)
- **Lab 5 integration**: Use the LLM-as-Judge pattern to evaluate the overall pipeline output quality
- **Domain labs**: Apply the Co-Scientist pattern to your specific research domain (BIO, HC, FIN, EOP labs)
- **Production**: Deploy with NeMo Agent Toolkit and NIM endpoints for scalable multi-agent inference

---
*Agentic AI Science Playbook — Foundation Layer, Lab 6.*